# Day 17 — Simplified: Telling the Model About ORDER

## The Surprising Problem

Attention has a weird property: **it doesn't know about position**.

```
"the cat sat"  ←→  "cat sat the"  ←→  "sat the cat"

To attention's math, these are the same. It sees a SET of tokens,
not a SEQUENCE.
```

That's obviously broken for language. "Dog bites man" and "Man bites dog" mean opposite things.

## Why Attention is Position-Blind

When attention computes scores, it does `Q_i · K_j`. Nothing in this formula uses positions `i` or `j`. If you shuffle the tokens, you get shuffled outputs — but each token's output is computed the same way.

## The Fix

Stamp each token with its position BEFORE feeding into attention.

```
word_at_position_0 = embedding("the") + position_vector(0)
word_at_position_1 = embedding("cat") + position_vector(1)
word_at_position_2 = embedding("sat") + position_vector(2)
```

Each token's representation now contains BOTH "what I am" AND "where I am."

## Two Ways To Make Position Vectors

### Way 1: Learned (GPT-2 style)

Just another `nn.Embedding`:
```python
self.tok_emb = nn.Embedding(vocab_size, embed_dim)   # what is this word
self.pos_emb = nn.Embedding(max_seq_len, embed_dim)  # where is this position
```

The model LEARNS what each position should look like during training.

### Way 2: Sinusoidal (Original Transformer)

Use a FIXED formula based on sines and cosines:
```
position 0 → [sin(0), cos(0), sin(0), cos(0), ...]
position 1 → [sin(1), cos(1), sin(1/100), cos(1/100), ...]
position 2 → [sin(2), cos(2), sin(2/100), cos(2/100), ...]
```

Different dimensions oscillate at different frequencies. Like a clock:
- Seconds hand: fast → distinguishes nearby positions
- Hour hand: slow → distinguishes far-apart positions

Together, they give every position a unique "barcode."

## Which Is Better?

Both work well. Modern models vary:
- GPT-2: learned positions
- Original Transformer: sinusoidal
- Llama: **RoPE** (newer trick — rotate Q and K by an angle proportional to position)

For us, learned is simplest. Sinusoidal is mathematically elegant.

## Where to Add Position Encoding

EXACTLY ONCE, right after the token embedding:

```python
x = tok_emb(idx) + pos_emb(positions)
# Then attention, attention, attention, attention...
```

Every attention layer downstream gets the position info for free.

## What This Means

Now your model can answer questions like:
- "Is this word at the start of the sentence?"
- "What was 5 tokens ago?"
- "Is the closing bracket too far back to remember?"

Without position encoding, none of those are possible. With it, your model can finally tell `"dog bites man"` from `"man bites dog"`.

## TL;DR

```
Self-attention sees a SET.
Position encoding turns it into a SEQUENCE.

input_with_position = token_embedding + position_embedding
```

That's the fix. One addition. The model now knows ORDER.

See `notebook.ipynb` for the proof-of-permutation-invariance demo + sinusoidal visualization.